In [13]:
import sys
import os
import urllib3
from configparser import ConfigParser

# Add your local ThreatConnect SDK to path
sys.path.append(r"Z:\HTOC\Data_Analytics\threatconnect")
from ThreatConnect import ThreatConnect
from RequestObject import RequestObject
from Owners import Owners

# Add your project repo to path
project_root = r"C:\Users\jaskew\Documents\HTOC\scripts\Data Movement\ThrearConnect-api-pull"
if project_root not in sys.path:
    sys.path.append(project_root)

from utils.config_loader import load_config

# Load API config
config_path = os.path.join(project_root, "utils", "config.json")
try:
    api_secret_key, api_access_id, api_base_url, api_default_org = load_config(config_path)
    display(f"Loaded config from: {config_path}")
    display(f"Base URL: {api_base_url}")
    display(f"Access ID: {api_access_id}")
    display(f"Default Org: {api_default_org}")
except Exception as e:
    display(f"[ERROR] Failed to load configuration: {e}")
    sys.exit(1)

# Disable SSL verification warnings (use cautiously)
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
verify_ssl = False

# Initialize ThreatConnect session
try:
    tc = ThreatConnect(api_access_id, api_secret_key, api_default_org, api_base_url)
    display("ThreatConnect initialized.")
except Exception as e:
    display(f"[ERROR] Failed to initialize ThreatConnect: {e}")
    sys.exit(1)

# Define the owner (organization scope)
owner = 'HTOC Org'

# Create a request object to fetch indicators (or other data)
try:
    ro = RequestObject()
    ro.set_http_method('GET')
    ro.set_owner(owner)
    ro.set_owner_allowed(True)
    # ro.set_resource_pagination(True)  # Uncomment if needed
    display("RequestObject successfully created.")
except Exception as e:
    display(f"[ERROR] Failed to initialize RequestObject: {e}")
    sys.exit(1)




'Loaded config from: C:\\Users\\jaskew\\Documents\\HTOC\\scripts\\Data Movement\\ThrearConnect-api-pull\\utils\\config.json'

'Base URL: https://hvs.threatconnect.com/api'

'Access ID: 09783848890162390382'

'Default Org: HTOC Org'

'ThreatConnect initialized.'

'RequestObject successfully created.'

In [14]:
import pandas as pd
from datetime import datetime, timedelta
import pytz
import urllib.parse

# Configuration for ThreatConnect indicator query
QUERY_LOOKBACK_HOURS = 48  # rolling wall-clock window in UTC (not calendar midnights)
INDICATOR_TYPE_NAMES = [
    "Address", "EmailAddress", "File", "Host", "URL", "ASN", "CIDR",
    "Email Subject", "Hashtag", "Mutex", "Registry Key", "User Agent",
]
OWNER_NAMES = [
    'HTOC Org',
    'CISA Federal Feed',
    'CMS_CTI',
    'Crowdstrike Falcon Intelligence',
    'DHS CISCP',
    'Intel471',
    'Mandiant Advantage Threat Intelligence',
    'VA_TIP Data',
]
RESULT_PAGE_SIZE = 500  # keep this smaller; same fields, just paged

# Single cutoff instant for TQL, observed_src filter, and workbook filter
_now_utc = datetime.now(pytz.UTC)
_last_observed_cutoff_dt = _now_utc - timedelta(hours=QUERY_LOOKBACK_HOURS)
start = _last_observed_cutoff_dt.strftime("%Y-%m-%dT%H:%M:%SZ")
LAST_OBSERVED_CUTOFF_TS = pd.Timestamp(_last_observed_cutoff_dt)

type_names = INDICATOR_TYPE_NAMES
type_name_condition = ", ".join([f'"{t}"' for t in type_names])

list_of_owners = OWNER_NAMES

# Build owner IN (...) clause
owner_condition = ", ".join([f'"{o}"' for o in list_of_owners])

tql_raw = (
    f'ownerName IN ({owner_condition}) AND '
    f'typeName IN ({type_name_condition}) AND '
    f'lastObserved >= "{start}"'
)

tql_encoded = urllib.parse.quote(tql_raw)

final_results = []

# Query indicators (paginate so you don't 502 with heavy fields)
# Create a NEW RequestObject WITHOUT owner restriction to query across all owners
ro_multi = RequestObject()
ro_multi.set_http_method('GET')

result_start = 0
result_limit = RESULT_PAGE_SIZE

while True:
    try:
        # NOTE: same fields list you requested (tags,observations,associatedGroups,falsePositives,threatAssess)
        # Only change here is removing the trailing comma after threatAssess which can break parsing.
        ro_multi.set_request_uri(
            f'/v3/indicators?tql={tql_encoded}'
            f'&fields=tags,observations,associatedGroups,falsePositives,threatAssess'
            f'&resultStart={result_start}&resultLimit={result_limit}'
        )

        response = tc.api_request(ro_multi)

        ct = response.headers.get('content-type', '')
        if not ct.startswith('application/json'):
            raise RuntimeError(f"Non-JSON response ({ct}): {response.content[:200]}")

        results = response.json()
        data_items = results.get('data', []) or []

        # stop when no more results
        if not data_items:
            break

        final_results.append(results)
        result_start += result_limit

    except Exception as e:
        display(f"Failed to query indicators (start={result_start}): {e}")
        break

# Normalize results
normalized_data = []
for result in final_results:
    data_items = result.get('data', [])
    if not data_items:
        display("No data returned in API response:", result)
    for item in data_items:
        if isinstance(item, dict) and 'summary' in item:
            normalized_data.append(item)

if normalized_data:
    observed_src = pd.json_normalize(normalized_data)
    observed_src['indicator'] = observed_src['summary'].astype(str).str.split().str[0].str.strip()
    observed_src['lastObserved'] = pd.to_datetime(observed_src['lastObserved'], utc=True, errors='coerce')
    observed_src = observed_src[observed_src["lastObserved"] >= LAST_OBSERVED_CUTOFF_TS]
    
    # Create a 'sources' column by aggregating ownerName values per indicator
    sources_per_indicator = (
        observed_src.groupby('indicator')['ownerName']
        .apply(lambda x: ', '.join(sorted(set(x))))
        .reset_index()
        .rename(columns={'ownerName': 'sources'})
    )

    # Merge sources back into observed_src
    observed_src = observed_src.merge(sources_per_indicator, on='indicator', how='left')
    # Filter to keep only records where ownerName is 'HTOC Org'
    observed_src = observed_src[observed_src['ownerName'] == 'HTOC Org'].copy()
    # Exclude indicators below threatAssessRating 3 and threatAssessConfidence < 50.
    # Rating: 1–5 scale (never threatAssess.rating — that is 0–1 normalized).
    # Confidence: 0–100 threatAssess fields only (not bare legacy 'confidence').
    # Coalesce flat vs nested columns per row so we do not drop rows where only one is populated.
    _rating_cols = ("threatAssessRating", "threatAssess.threatAssessRating")
    _confidence_cols = ("threatAssessConfidence", "threatAssess.threatAssessConfidence")

    def _first_non_null_numeric(df, ordered_cols):
        present = [c for c in ordered_cols if c in df.columns]
        if not present:
            return None
        out = pd.to_numeric(df[present[0]], errors="coerce")
        for c in present[1:]:
            s = pd.to_numeric(df[c], errors="coerce")
            out = out.mask(out.isna(), s)
        return out

    _ta = _first_non_null_numeric(observed_src, _rating_cols)
    _tc = _first_non_null_numeric(observed_src, _confidence_cols)
    if _ta is None or _tc is None:
        raise KeyError(
            f"Could not resolve Threat Assess columns. Tried rating={_rating_cols}, "
            f"confidence={_confidence_cols}. Columns: {list(observed_src.columns)}"
        )
    # Some rows store 0–1 normalized threat in threatAssessRating (e.g. 0.75) while the
    # top-level `rating` column holds the 1–5 band (e.g. 3.0).
    if "rating" in observed_src.columns:
        _r = pd.to_numeric(observed_src["rating"], errors="coerce")
        _use_band = _ta.notna() & (_ta < 1.0) & _r.notna() & (_r >= 1) & (_r <= 5)
        _ta = _ta.where(~_use_band, _r)
    _pre_ta = len(observed_src)
    # Use >= 50 so a boundary value of 50.0 is included (strict > 50 dropped those rows).
    observed_src = observed_src[(_ta >= 3) & (_tc >= 50)].copy()
    display(
        f"Threat assess filter (1–5 rating>=3, confidence>=50) coalescing {_rating_cols} / {_confidence_cols} "
        f"(+ `rating` when TA value is 0–1 normalized): {_pre_ta} -> {len(observed_src)} rows."
    )
else:
    display("No valid indicator data found.")
    observed_src = pd.DataFrame()

display(observed_src)

"Threat assess filter (1–5 rating>=3, confidence>=50) coalescing ('threatAssessRating', 'threatAssess.threatAssessRating') / ('threatAssessConfidence', 'threatAssess.threatAssessConfidence') (+ `rating` when TA value is 0–1 normalized): 769 -> 159 rows."

,id,dateAdded,ownerId,ownerName,webLink,type,lastModified,rating,confidence,threatAssessRating,...,ip,legacyLink,tags.data,hostName,dnsActive,whoisActive,associatedGroups.data,source,indicator,sources
4,5629499589020828,2026-02-25T13:50:36Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2026-04-20T11:19:31Z,3.0,78.0,3.0,...,172.105.147.48,https://hvs.threatconnect.com/auth/indicators/...,"[{'id': 2076981, 'name': 'SOAR Indicator PB', ...",NaN,NaN,NaN,NaN,NaN,172.105.147.48,HTOC Org
22,6755399465229542,2025-07-28T17:34:22Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2026-04-20T11:19:27Z,3.0,67.0,3.0,...,64.62.156.210,https://hvs.threatconnect.com/auth/indicators/...,"[{'id': 2076981, 'name': 'SOAR Indicator PB', ...",NaN,NaN,NaN,NaN,NaN,64.62.156.210,HTOC Org
24,6755399467420966,2025-08-13T15:08:20Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2026-04-20T11:19:26Z,3.0,50.0,3.0,...,65.49.1.97,https://hvs.threatconnect.com/auth/indicators/...,"[{'id': 2076981, 'name': 'SOAR Indicator PB', ...",NaN,NaN,NaN,NaN,NaN,65.49.1.97,HTOC Org
100,5629499567010841,2025-09-05T17:42:55Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2026-04-20T11:19:12Z,3.0,52.0,3.0,...,35.203.210.74,https://hvs.threatconnect.com/auth/indicators/...,"[{'id': 2076981, 'name': 'SOAR Indicator PB', ...",NaN,NaN,NaN,NaN,NaN,35.203.210.74,HTOC Org
107,5629499537015531,2025-04-23T15:01:06Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2026-04-20T10:19:13Z,5.0,50.0,5.0,...,146.70.196.195,https://hvs.threatconnect.com/auth/indicators/...,"[{'id': 2023876, 'name': 'VA OIT CSOC CTS Splu...",NaN,NaN,NaN,"[{'id': 6755399444000513, 'dateAdded': '2025-0...",NaN,146.70.196.195,HTOC Org
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
907,5629499574089574,2025-10-21T11:33:56Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2026-04-13T14:19:18Z,5.0,76.0,4.0,...,70.39.70.194,https://hvs.threatconnect.com/auth/indicators/...,"[{'id': 1739147, 'name': 'OtterCookie', 'lastU...",NaN,NaN,NaN,"[{'id': 6755399470002411, 'dateAdded': '2025-1...",NaN,70.39.70.194,"HTOC Org, Mandiant Advantage Threat Intelligence"
918,5629499537015519,2025-04-23T15:01:06Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2026-04-13T14:19:12Z,5.0,51.0,5.0,...,138.199.6.195,https://hvs.threatconnect.com/auth/indicators/...,"[{'id': 2023876, 'name': 'VA OIT CSOC CTS Splu...",NaN,NaN,NaN,"[{'id': 6755399444000513, 'dateAdded': '2025-0...",NaN,138.199.6.195,HTOC Org
929,6755399482138722,2025-11-25T18:08:22Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2026-04-13T14:12:39Z,3.0,61.0,3.0,...,195.96.138.145,https://hvs.threatconnect.com/auth/indicators/...,"[{'id': 2076981, 'name': 'SOAR Indicator PB', ...",NaN,NaN,NaN,NaN,NaN,195.96.138.145,HTOC Org
935,6755399488041556,2026-01-22T15:37:51Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2026-04-13T12:31:32Z,3.0,69.0,3.0,...,44.192.244.62,https://hvs.threatconnect.com/auth/indicators/...,"[{'id': 2076981, 'name': 'SOAR Indicator PB', ...",NaN,NaN,NaN,NaN,NaN,44.192.244.62,HTOC Org


In [15]:
observed_src[observed_src['indicator'] == '165.154.179.204']

,id,dateAdded,ownerId,ownerName,webLink,type,lastModified,rating,confidence,threatAssessRating,...,ip,legacyLink,tags.data,hostName,dnsActive,whoisActive,associatedGroups.data,source,indicator,sources
375,6755399459078388,2025-06-25T13:41:27Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2026-04-20T09:00:56Z,3.0,58.0,0.75,...,165.154.179.204,https://hvs.threatconnect.com/auth/indicators/...,"[{'id': 2076981, 'name': 'SOAR Indicator PB', ...",NaN,NaN,NaN,"[{'id': 5629499581002461, 'dateAdded': '2026-0...",NaN,165.154.179.204,"CMS_CTI, DHS CISCP, HTOC Org"


In [ ]:
import pandas as pd
import ast
from datetime import datetime, timedelta
import pytz

# Load the Excel file
file_path = r"Z:\HTOC\Data_Analytics\Data\Threat Assessment Scores\Threat_Assessment_Scores.xlsx"
df = pd.read_excel(file_path)


# Keep only indicators that are also in observed_src
_indicator_col = next((c for c in ["indicator", "Indicator", "INDICATOR"] if c in df.columns), None)
if _indicator_col is None:
    raise KeyError(f"Could not find indicator column in df. Columns: {list(df.columns)}")

_observed_indicators = set(observed_src["indicator"].dropna().astype(str))
df = df[df[_indicator_col].astype(str).isin(_observed_indicators)].copy()

# Last Observed column: values come only from observed_src (ThreatConnect), not the workbook
_last_observed_col = next(
    (
        c
        for c in [
            "Last Observed",
            "lastObserved",
            "LastObserved",
            "last_observed",
            "LAST OBSERVED",
        ]
        if c in df.columns
    ),
    None,
)
if _last_observed_col is None:
    raise KeyError(f"Could not find 'Last Observed' column in df. Columns: {list(df.columns)}")

_assoc_groups_src_col = "associatedGroups.data"
_assoc_groups_target_col = "Associated Groups"
if _assoc_groups_src_col not in observed_src.columns:
    raise KeyError(
        f"Could not find '{_assoc_groups_src_col}' column in observed_src. Columns: {list(observed_src.columns)}"
    )


def _extract_group_ids(value):
    # Handle scalar nulls safely; avoid pd.isna on list-like values.
    if value is None:
        return pd.NA

    parsed = value
    if isinstance(value, str):
        text = value.strip()
        if not text:
            return pd.NA
        try:
            parsed = ast.literal_eval(text)
        except (ValueError, SyntaxError):
            return text
    elif isinstance(value, float) and pd.isna(value):
        return pd.NA

    if isinstance(parsed, dict):
        gid = parsed.get("id")
        return f"Group Id: {gid}" if gid is not None else pd.NA

    if isinstance(parsed, list):
        ids = []
        for item in parsed:
            if isinstance(item, dict) and item.get("id") is not None:
                ids.append(f"Group Id: {item.get('id')}")
        return ", ".join(ids) if ids else pd.NA

    return pd.NA


_observed_latest = (
    observed_src.dropna(subset=["indicator"])
    .assign(
        indicator=lambda d: d["indicator"].astype(str),
        lastObserved=lambda d: pd.to_datetime(d["lastObserved"], utc=True, errors="coerce"),
    )
    .sort_values("lastObserved")
    .drop_duplicates(subset=["indicator"], keep="last")
)

_last_obs_by_indicator = _observed_latest.set_index("indicator")["lastObserved"]
_assoc_groups_by_indicator = _observed_latest.set_index("indicator")[_assoc_groups_src_col].map(_extract_group_ids)

# Last Observed: only from ThreatConnect (observed_src); do not fall back to Excel dates
_df_ind = df[_indicator_col].astype(str)
df[_last_observed_col] = pd.to_datetime(_df_ind.map(_last_obs_by_indicator), utc=True, errors="coerce")
_qh = QUERY_LOOKBACK_HOURS if "QUERY_LOOKBACK_HOURS" in globals() else 48
_last_obs_cutoff = (
    LAST_OBSERVED_CUTOFF_TS
    if "LAST_OBSERVED_CUTOFF_TS" in globals()
    else pd.Timestamp(datetime.now(pytz.UTC) - timedelta(hours=_qh), tz="UTC")
)
_pre_lo = len(df)
df = df[df[_last_observed_col].notna() & (df[_last_observed_col] >= _last_obs_cutoff)].copy()
display(
    f"Last Observed filter (ThreatConnect only, >= {_last_obs_cutoff}): {_pre_lo} -> {len(df)} rows."
)

_df_ind = df[_indicator_col].astype(str)

# Add associatedGroups.data ids from observed_src by indicator, stored as 'Associated Groups'
if _assoc_groups_target_col in df.columns:
    df[_assoc_groups_target_col] = _df_ind.map(_assoc_groups_by_indicator).combine_first(df[_assoc_groups_target_col])
else:
    df[_assoc_groups_target_col] = _df_ind.map(_assoc_groups_by_indicator)

df

BadZipFile: File is not a zip file

In [ ]:
import os
import pandas as pd
from datetime import datetime, timedelta

# Base file path with placeholder for date
base_path = r"Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d{date}.csv"
#base_path = r"C:\Users\jaskew\Documents\project_repository\data\raw\ObservationDataFiles\htoc_opdiv_obs_d{date}.csv"
date_format = "%Y%m%d"

def get_file_paths(base_path, days=3):
    """ Generate file paths for the last `days` days using list comprehension. """
    today = datetime.utcnow()
    dates_to_pull = [(today - timedelta(days=i)).strftime(date_format) for i in range(days)]
    
    # Construct file paths
    file_paths = [base_path.format(date=dt) for dt in dates_to_pull]
    
    # Filter for existing files
    existing_files = [file_path for file_path in file_paths if os.path.exists(file_path)]
    
    if not existing_files:
        print("No files found for the specified date range.")
    else:
        print(f"Files to be loaded: {existing_files}")
    
    return existing_files

def load_observed_data(file_paths):
    """ Load and concatenate observed data from multiple files. """
    data_frames = []

    for file_path in file_paths:
        try:
            df = pd.read_csv(file_path)
            data_frames.append(df)
        except Exception as e:
            print(f"Error reading file {file_path}: {e}")
    
    # Concatenate data
    if data_frames:
        observed_data_df = pd.concat(data_frames, ignore_index=True)
        print(f"Loaded data from {len(data_frames)} files.")
    else:
        observed_data_df = pd.DataFrame()

    return observed_data_df

# Example Usage:
# Fetch file paths for the last 3 days
file_paths = get_file_paths(base_path, days=2)

# Load observed data
observed_data_df = load_observed_data(file_paths)



Files to be loaded: ['Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20260420.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20260419.csv']
Loaded data from 2 files.


C:\Users\jaskew\AppData\Local\Temp\ipykernel_28180\1484081131.py:12: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  today = datetime.utcnow()


In [ ]:
# Keep only indicators that are present in observed_data_df and seen by 2+ OpDiv partners
_indicator_col_df = next((c for c in ["indicator", "Indicator", "INDICATOR"] if c in df.columns), None)
_indicator_col_obs = next((c for c in ["indicator", "Indicator", "INDICATOR"] if c in observed_data_df.columns), None)
_opdiv_col = next((c for c in ["OpDiv", "opdiv", "OPDIV"] if c in observed_data_df.columns), None)

if _indicator_col_df is None:
    raise KeyError(f"Could not find indicator column in df. Columns: {list(df.columns)}")
if _indicator_col_obs is None:
    raise KeyError(f"Could not find indicator column in observed_data_df. Columns: {list(observed_data_df.columns)}")
if _opdiv_col is None:
    raise KeyError(f"Could not find OpDiv column in observed_data_df. Columns: {list(observed_data_df.columns)}")

obs = observed_data_df.dropna(subset=[_indicator_col_obs, _opdiv_col]).copy()
obs[_indicator_col_obs] = obs[_indicator_col_obs].astype(str).str.strip()
obs[_opdiv_col] = obs[_opdiv_col].astype(str).str.strip()

partners_by_indicator = (
    obs.groupby(_indicator_col_obs)[_opdiv_col]
    .apply(lambda s: sorted(set(x for x in s if x)))
)

eligible_partners = partners_by_indicator[partners_by_indicator.str.len() >= 2]
opdiv_map = eligible_partners.apply(lambda vals: ", ".join(vals))

last_24h_multiple_partners = df[
    df[_indicator_col_df].astype(str).str.strip().isin(eligible_partners.index)
].copy()
last_24h_multiple_partners["OpDiv"] = (
    last_24h_multiple_partners[_indicator_col_df].astype(str).str.strip().map(opdiv_map)
)
last_24h_multiple_partners["Partners"] = last_24h_multiple_partners["OpDiv"]

last_24h_multiple_partners

,Indicator,Last Observed,Indicator Type,Observation Yearly Count,ThreatConnect Rating,Observation Penalty Multiplier,Botnet Flag,False Positives,Partners,incidents/events,Threat Actor,CAL Score,ThreatConnect Score,PRISM Score,Severity,Explanation,Associated Groups,OpDiv
3,180.167.128.202,2026-04-20 00:00:00+00:00,Address,33,3,0.998192,0,0,"CDC, CMS, FDA, HRSA, IHS, NIH, OS",Incident:INC9444954,NaN,180,527,691,high,[2026-04-13] Severity: high. VT score: 13. Con...,<NA>,"CDC, CMS, FDA, HRSA, IHS, NIH, OS"
12,64.23.214.73,2026-04-20 00:00:00+00:00,Address,124,3,0.993205,0,0,"FDA, OS, VA",Incident:INC9328182,NaN,470,614,482,medium,[2026-04-13] Severity: medium. VT score: 7. Co...,<NA>,"FDA, OS, VA"
15,80.15.177.203,2026-04-20 00:00:00+00:00,Address,116,3,0.993644,0,0,"CMS, FDA, IHS, OS, VA",Incident:INC9341092,NaN,180,469,346,medium,[2026-04-13] Severity: medium. VT score: 6. Co...,<NA>,"CMS, FDA, IHS, OS, VA"
17,91.224.92.147,2026-04-20 00:00:00+00:00,Address,20,3,0.998904,0,0,"CDC, CMS, FDA, HRSA, IHS, NIH, OS, VA",Incident:INC9480324,NaN,180,590,484,medium,[2026-04-13] Severity: medium. VT score: 12. C...,<NA>,"CDC, CMS, FDA, HRSA, IHS, NIH, OS, VA"
39,103.120.116.162,2026-04-20 00:00:00+00:00,Address,76,3,0.995836,0,0,"CMS, FDA, OS, VA",Incident:INC9385644,NaN,730,865,677,high,[2026-04-13] Severity: high. VT score: 12. Con...,<NA>,"CMS, FDA, OS, VA"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2187,87.251.64.158,2026-04-20 00:00:00+00:00,Address,4,3,0.999781,0,0,"CDC, CMS, FDA, HRSA, IHS, NIH, OS, VA",Incident:INC9486336;Incident:INC9486332,NaN,430,715,466,medium,[2026-04-13] Severity: medium. VT score: 5. Co...,<NA>,"CDC, CMS, FDA, HRSA, IHS, NIH, OS, VA"
2230,89.190.156.34,2026-04-20 00:00:00+00:00,Address,1,3,0.999945,0,0,"HHS, NIH",NaN,NaN,870,633,587,high,[2026-04-13] Severity: high. VT score: 16. Con...,<NA>,"HHS, NIH"
2241,91.196.152.88,2026-04-20 00:00:00+00:00,Address,95,3,0.994795,0,0,"CMS, FDA, NIH, OS",NaN,NaN,180,590,269,medium,[2026-04-13] Severity: medium. VT score: 7. Co...,<NA>,"CMS, FDA, NIH, OS"
2266,93.100.91.16,2026-04-20 00:00:00+00:00,Address,153,3,0.991616,0,0,"CMS, FDA, HHS, OS",Incident:INC9282086,NaN,180,469,344,medium,[2026-04-13] Severity: medium. VT score: 6. Co...,<NA>,"CMS, FDA, HHS, OS"


In [ ]:
# Filter multi-partner, last-24h indicators to VT score >= 10 based on Explanation text
vt_scores = last_24h_multiple_partners['Explanation'].str.extract(r'VT score:\s*(\d+)', expand=False)
vt_scores = pd.to_numeric(vt_scores, errors='coerce')

last_24h_multi_partners_vt15 = last_24h_multiple_partners[vt_scores >= 14]

last_24h_multi_partners_vt15

,Indicator,Last Observed,Indicator Type,Observation Yearly Count,ThreatConnect Rating,Observation Penalty Multiplier,Botnet Flag,False Positives,Partners,incidents/events,Threat Actor,CAL Score,ThreatConnect Score,PRISM Score,Severity,Explanation,Associated Groups,OpDiv
541,165.154.179.204,2026-04-20 00:00:00+00:00,Address,279,3,0.984712,0,0,"CMS, FDA, OS",Incident:INC9118016,NaN,550,531,793,high,[2026-04-13] Severity: critical. VT score: 14....,Group Id: 5629499581002461,"CMS, FDA, OS"
822,185.226.197.47,2026-04-20 00:00:00+00:00,Address,90,3,0.995068,0,0,"CMS, FDA, HRSA, NIH, OS, VA",Incident:INC9366814,NaN,470,672,715,high,[2026-04-13] Severity: high. VT score: 14. Con...,<NA>,"CMS, FDA, HRSA, NIH, OS, VA"
824,185.226.197.50,2026-04-20 00:00:00+00:00,Address,90,3,0.995068,0,0,"CMS, FDA, HRSA, IHS, NIH, OS, VA",Incident:INC9366814,NaN,460,667,714,high,[2026-04-13] Severity: high. VT score: 14. Con...,<NA>,"CMS, FDA, HRSA, IHS, NIH, OS, VA"
1061,195.178.110.188,2026-04-20 00:00:00+00:00,Address,7,3,0.999616,0,0,"CMS, IHS, NIH, OS",Incident:INC9480324,NaN,180,590,544,high,[2026-04-13] Severity: high. VT score: 14. Con...,<NA>,"CMS, IHS, NIH, OS"
1637,45.153.34.87,2026-04-20 00:00:00+00:00,Address,62,3,0.996603,0,0,"CMS, FDA, HRSA, IHS, OS, VA",Incident:INC9366814,NaN,220,610,720,high,[2026-04-13] Severity: high. VT score: 16. Con...,<NA>,"CMS, FDA, HRSA, IHS, OS, VA"
1647,45.230.66.104,2026-04-20 00:00:00+00:00,Address,148,3,0.991890,0,0,"CMS, FDA, HRSA, OS, VA",NaN,NaN,180,507,596,high,[2026-04-13] Severity: high. VT score: 14. Con...,<NA>,"CMS, FDA, HRSA, OS, VA"
1744,61.182.2.26,2026-04-20 00:00:00+00:00,Address,7,3,0.999616,0,0,"CMS, FDA, HRSA, NIH, OS",Incident:INC9480324,NaN,800,837,564,high,[2026-04-13] Severity: high. VT score: 15. Con...,<NA>,"CMS, FDA, HRSA, NIH, OS"
1943,65.49.1.108,2026-04-20 00:00:00+00:00,Address,293,3,0.983945,0,0,"CMS, FDA, HHS, HRSA, IHS, OS, VA",NaN,NaN,180,469,595,high,[2026-04-13] Severity: high. VT score: 14. Con...,<NA>,"CMS, FDA, HHS, HRSA, IHS, OS, VA"
2230,89.190.156.34,2026-04-20 00:00:00+00:00,Address,1,3,0.999945,0,0,"HHS, NIH",NaN,NaN,870,633,587,high,[2026-04-13] Severity: high. VT score: 16. Con...,<NA>,"HHS, NIH"


In [ ]:
# Keep only high or critical indicators from the VT>=10, multi-partner, last-24h set
final_indicators = last_24h_multi_partners_vt15[last_24h_multi_partners_vt15['Severity'].isin(['high', 'critical'])]

final_indicators

,Indicator,Last Observed,Indicator Type,Observation Yearly Count,ThreatConnect Rating,Observation Penalty Multiplier,Botnet Flag,False Positives,Partners,incidents/events,Threat Actor,CAL Score,ThreatConnect Score,PRISM Score,Severity,Explanation,Associated Groups,OpDiv
541,165.154.179.204,2026-04-20 00:00:00+00:00,Address,279,3,0.984712,0,0,"CMS, FDA, OS",Incident:INC9118016,NaN,550,531,793,high,[2026-04-13] Severity: critical. VT score: 14....,Group Id: 5629499581002461,"CMS, FDA, OS"
822,185.226.197.47,2026-04-20 00:00:00+00:00,Address,90,3,0.995068,0,0,"CMS, FDA, HRSA, NIH, OS, VA",Incident:INC9366814,NaN,470,672,715,high,[2026-04-13] Severity: high. VT score: 14. Con...,<NA>,"CMS, FDA, HRSA, NIH, OS, VA"
824,185.226.197.50,2026-04-20 00:00:00+00:00,Address,90,3,0.995068,0,0,"CMS, FDA, HRSA, IHS, NIH, OS, VA",Incident:INC9366814,NaN,460,667,714,high,[2026-04-13] Severity: high. VT score: 14. Con...,<NA>,"CMS, FDA, HRSA, IHS, NIH, OS, VA"
1061,195.178.110.188,2026-04-20 00:00:00+00:00,Address,7,3,0.999616,0,0,"CMS, IHS, NIH, OS",Incident:INC9480324,NaN,180,590,544,high,[2026-04-13] Severity: high. VT score: 14. Con...,<NA>,"CMS, IHS, NIH, OS"
1637,45.153.34.87,2026-04-20 00:00:00+00:00,Address,62,3,0.996603,0,0,"CMS, FDA, HRSA, IHS, OS, VA",Incident:INC9366814,NaN,220,610,720,high,[2026-04-13] Severity: high. VT score: 16. Con...,<NA>,"CMS, FDA, HRSA, IHS, OS, VA"
1647,45.230.66.104,2026-04-20 00:00:00+00:00,Address,148,3,0.991890,0,0,"CMS, FDA, HRSA, OS, VA",NaN,NaN,180,507,596,high,[2026-04-13] Severity: high. VT score: 14. Con...,<NA>,"CMS, FDA, HRSA, OS, VA"
1744,61.182.2.26,2026-04-20 00:00:00+00:00,Address,7,3,0.999616,0,0,"CMS, FDA, HRSA, NIH, OS",Incident:INC9480324,NaN,800,837,564,high,[2026-04-13] Severity: high. VT score: 15. Con...,<NA>,"CMS, FDA, HRSA, NIH, OS"
1943,65.49.1.108,2026-04-20 00:00:00+00:00,Address,293,3,0.983945,0,0,"CMS, FDA, HHS, HRSA, IHS, OS, VA",NaN,NaN,180,469,595,high,[2026-04-13] Severity: high. VT score: 14. Con...,<NA>,"CMS, FDA, HHS, HRSA, IHS, OS, VA"
2230,89.190.156.34,2026-04-20 00:00:00+00:00,Address,1,3,0.999945,0,0,"HHS, NIH",NaN,NaN,870,633,587,high,[2026-04-13] Severity: high. VT score: 16. Con...,<NA>,"HHS, NIH"


In [ ]:
import pandas as pd

# Load external tags data
tags_path = r"Z:\HTOC\Data_Analytics\Data\Observed_Tags\htoc_observed_indicator_tags.csv"
tags_df = pd.read_csv(tags_path)

# The indicator column in the tags CSV could be e.g. 'Indicator' or 'indicator'
tags_indicator_col = None
for col in tags_df.columns:
    if str(col).lower() == 'indicator':
        tags_indicator_col = col
        break
if tags_indicator_col is None:
    raise ValueError("Could not find an 'Indicator' column in the tags CSV.")

# The tags field might be called 'Tags', 'tags', or similar
# The tags field might be called 'Tags', 'tags', 'Tag', 'tag', etc.
tags_value_col = None
for col in tags_df.columns:
    if str(col).lower() in ('tags', 'tag'):
        tags_value_col = col
        break
if tags_value_col is None:
    raise ValueError(
        f"Could not find a 'Tag' or 'Tags' column in the tags CSV. "
        f"Available columns: {list(tags_df.columns)}"
    )
# For fast lookup, set up a mapping of indicator -> tags value.
indicator_to_tags = tags_df.set_index(tags_indicator_col)[tags_value_col].to_dict()

# Prepare 'Tags' values for final_indicators
final_tags = final_indicators['Indicator'].map(indicator_to_tags)

# Insert the 'Tags' column as the second to last column
final_cols = list(final_indicators.columns)
if 'Tags' in final_cols:
    final_cols.remove('Tags')
second_to_last_idx = -1 if len(final_cols) == 0 else -1
new_cols = final_cols[:second_to_last_idx] + ['Tags'] + final_cols[second_to_last_idx:]

final_indicators['Tags'] = final_tags
final_indicators = final_indicators[new_cols]
final_indicators


,Indicator,Last Observed,Indicator Type,Observation Yearly Count,ThreatConnect Rating,Observation Penalty Multiplier,Botnet Flag,False Positives,Partners,incidents/events,Threat Actor,CAL Score,ThreatConnect Score,PRISM Score,Severity,Explanation,Associated Groups,Tags,OpDiv
541,165.154.179.204,2026-04-20 00:00:00+00:00,Address,279,3,0.984712,0,0,"CMS, FDA, OS",Incident:INC9118016,NaN,550,531,793,high,[2026-04-13] Severity: critical. VT score: 14....,Group Id: 5629499581002461,Brute Force,"CMS, FDA, OS"
822,185.226.197.47,2026-04-20 00:00:00+00:00,Address,90,3,0.995068,0,0,"CMS, FDA, HRSA, NIH, OS, VA",Incident:INC9366814,NaN,470,672,715,high,[2026-04-13] Severity: high. VT score: 14. Con...,<NA>,Web Scanner,"CMS, FDA, HRSA, NIH, OS, VA"
824,185.226.197.50,2026-04-20 00:00:00+00:00,Address,90,3,0.995068,0,0,"CMS, FDA, HRSA, IHS, NIH, OS, VA",Incident:INC9366814,NaN,460,667,714,high,[2026-04-13] Severity: high. VT score: 14. Con...,<NA>,Web Scanner,"CMS, FDA, HRSA, IHS, NIH, OS, VA"
1061,195.178.110.188,2026-04-20 00:00:00+00:00,Address,7,3,0.999616,0,0,"CMS, IHS, NIH, OS",Incident:INC9480324,NaN,180,590,544,high,[2026-04-13] Severity: high. VT score: 14. Con...,<NA>,Web Scanner,"CMS, IHS, NIH, OS"
1637,45.153.34.87,2026-04-20 00:00:00+00:00,Address,62,3,0.996603,0,0,"CMS, FDA, HRSA, IHS, OS, VA",Incident:INC9366814,NaN,220,610,720,high,[2026-04-13] Severity: high. VT score: 16. Con...,<NA>,Web Scanner,"CMS, FDA, HRSA, IHS, OS, VA"
1647,45.230.66.104,2026-04-20 00:00:00+00:00,Address,148,3,0.991890,0,0,"CMS, FDA, HRSA, OS, VA",NaN,NaN,180,507,596,high,[2026-04-13] Severity: high. VT score: 14. Con...,<NA>,Web Scanner,"CMS, FDA, HRSA, OS, VA"
1744,61.182.2.26,2026-04-20 00:00:00+00:00,Address,7,3,0.999616,0,0,"CMS, FDA, HRSA, NIH, OS",Incident:INC9480324,NaN,800,837,564,high,[2026-04-13] Severity: high. VT score: 15. Con...,<NA>,Web Scanner,"CMS, FDA, HRSA, NIH, OS"
1943,65.49.1.108,2026-04-20 00:00:00+00:00,Address,293,3,0.983945,0,0,"CMS, FDA, HHS, HRSA, IHS, OS, VA",NaN,NaN,180,469,595,high,[2026-04-13] Severity: high. VT score: 14. Con...,<NA>,Web Scanner,"CMS, FDA, HHS, HRSA, IHS, OS, VA"
2230,89.190.156.34,2026-04-20 00:00:00+00:00,Address,1,3,0.999945,0,0,"HHS, NIH",NaN,NaN,870,633,587,high,[2026-04-13] Severity: high. VT score: 16. Con...,<NA>,SOAR Indicator PB,"HHS, NIH"


In [ ]:
import pandas as pd

# Helper to see if an indicator has an I&W tag
def has_iw(tags_value):
    """
    tags_value is typically a list of dicts from ThreatConnect, e.g.:
    [{'name': 'I&W'}, {'name': 'something else'}, ...]
    """
    if tags_value is None or (isinstance(tags_value, float) and pd.isna(tags_value)):
        return False

    if not isinstance(tags_value, (list, tuple)):
        return False

    for t in tags_value:
        try:
            if isinstance(t, dict):
                name = str(t.get('name', '')).strip()
            else:
                name = str(t).strip()

            if name.lower() in {"i&w", "i & w", "iw"}:
                return True
        except Exception:
            continue
    return False

# 1) Add has_iw flag to observed_src if tags.data exists
if 'tags.data' in observed_src.columns:
    observed_src['has_iw'] = observed_src['tags.data'].apply(has_iw)
else:
    observed_src['has_iw'] = False

# 2) Collapse to one flag per indicator
iw_per_indicator = (
    observed_src.groupby('indicator', dropna=False)['has_iw']
    .max()  # any True -> True
    .reset_index()
    .rename(columns={'indicator': 'Indicator', 'has_iw': 'Reported I&W?_raw'})
)

# 3) Drop ANY existing Reported I&W? variants (_x, _y, etc.)
cols_to_drop = [c for c in final_indicators.columns if c.startswith('Reported I&W?')]
final_indicators = final_indicators.drop(columns=cols_to_drop, errors='ignore')

# 4) Merge once, with a temporary raw boolean column
final_indicators = final_indicators.merge(
    iw_per_indicator,
    on='Indicator',
    how='left'
)

# 5) Convert to Yes/No, defaulting missing to 'No'
final_indicators['Reported I&W?'] = (
    final_indicators['Reported I&W?_raw']
    .fillna(False)
    .map({True: 'Yes', False: 'No'})
)

# 6) Drop the temporary raw column
final_indicators = final_indicators.drop(columns=['Reported I&W?_raw'])

# Rename column 'HTOC Threat Score' to 'PRISM Score' if it exists
if "HTOC Threat Score" in final_indicators.columns:
    final_indicators = final_indicators.rename(columns={"HTOC Threat Score": "PRISM Score"})


final_indicators

,Indicator,Last Observed,Indicator Type,Observation Yearly Count,ThreatConnect Rating,Observation Penalty Multiplier,Botnet Flag,False Positives,Partners,incidents/events,Threat Actor,CAL Score,ThreatConnect Score,PRISM Score,Severity,Explanation,Associated Groups,Tags,OpDiv,Reported I&W?
0,165.154.179.204,2026-04-20 00:00:00+00:00,Address,279,3,0.984712,0,0,"CMS, FDA, OS",Incident:INC9118016,NaN,550,531,793,high,[2026-04-13] Severity: critical. VT score: 14....,Group Id: 5629499581002461,Brute Force,"CMS, FDA, OS",Yes
1,185.226.197.47,2026-04-20 00:00:00+00:00,Address,90,3,0.995068,0,0,"CMS, FDA, HRSA, NIH, OS, VA",Incident:INC9366814,NaN,470,672,715,high,[2026-04-13] Severity: high. VT score: 14. Con...,<NA>,Web Scanner,"CMS, FDA, HRSA, NIH, OS, VA",No
2,185.226.197.50,2026-04-20 00:00:00+00:00,Address,90,3,0.995068,0,0,"CMS, FDA, HRSA, IHS, NIH, OS, VA",Incident:INC9366814,NaN,460,667,714,high,[2026-04-13] Severity: high. VT score: 14. Con...,<NA>,Web Scanner,"CMS, FDA, HRSA, IHS, NIH, OS, VA",No
3,195.178.110.188,2026-04-20 00:00:00+00:00,Address,7,3,0.999616,0,0,"CMS, IHS, NIH, OS",Incident:INC9480324,NaN,180,590,544,high,[2026-04-13] Severity: high. VT score: 14. Con...,<NA>,Web Scanner,"CMS, IHS, NIH, OS",No
4,45.153.34.87,2026-04-20 00:00:00+00:00,Address,62,3,0.996603,0,0,"CMS, FDA, HRSA, IHS, OS, VA",Incident:INC9366814,NaN,220,610,720,high,[2026-04-13] Severity: high. VT score: 16. Con...,<NA>,Web Scanner,"CMS, FDA, HRSA, IHS, OS, VA",No
5,45.230.66.104,2026-04-20 00:00:00+00:00,Address,148,3,0.991890,0,0,"CMS, FDA, HRSA, OS, VA",NaN,NaN,180,507,596,high,[2026-04-13] Severity: high. VT score: 14. Con...,<NA>,Web Scanner,"CMS, FDA, HRSA, OS, VA",No
6,61.182.2.26,2026-04-20 00:00:00+00:00,Address,7,3,0.999616,0,0,"CMS, FDA, HRSA, NIH, OS",Incident:INC9480324,NaN,800,837,564,high,[2026-04-13] Severity: high. VT score: 15. Con...,<NA>,Web Scanner,"CMS, FDA, HRSA, NIH, OS",No
7,65.49.1.108,2026-04-20 00:00:00+00:00,Address,293,3,0.983945,0,0,"CMS, FDA, HHS, HRSA, IHS, OS, VA",NaN,NaN,180,469,595,high,[2026-04-13] Severity: high. VT score: 14. Con...,<NA>,Web Scanner,"CMS, FDA, HHS, HRSA, IHS, OS, VA",No
8,89.190.156.34,2026-04-20 00:00:00+00:00,Address,1,3,0.999945,0,0,"HHS, NIH",NaN,NaN,870,633,587,high,[2026-04-13] Severity: high. VT score: 16. Con...,<NA>,SOAR Indicator PB,"HHS, NIH",No


In [ ]:
from datetime import datetime

# Build dated output path
today_str = datetime.today().strftime('%Y%m%d')  # e.g. 20260316
output_path = rf"Z:\HTOC\Data_Analytics\Data\Threat Assessment Scores\ThreatAssessI_W\ThreatAssessI_W_{today_str}.xlsx"

# Excel can't write timezone-aware datetimes; strip tz info before export
_dt_tz_cols = final_indicators.select_dtypes(include=["datetimetz"]).columns
for _c in _dt_tz_cols:
    final_indicators[_c] = final_indicators[_c].dt.tz_convert(None)

iw_col = "Reported I&W?"
if iw_col not in final_indicators.columns:
    raise KeyError(f"Missing required column '{iw_col}' for sheet split.")

final_iw_no = final_indicators[final_indicators[iw_col] == "No"].copy()
final_iw_yes = final_indicators[final_indicators[iw_col] == "Yes"].copy()

# Write to one workbook with two named sheets
with pd.ExcelWriter(output_path, engine="xlsxwriter") as writer:
    final_iw_no.to_excel(writer, index=False, sheet_name="I&W_No")
    final_iw_yes.to_excel(writer, index=False, sheet_name="I&W_Yes")

    workbook = writer.book
    wrap_fmt = workbook.add_format({"text_wrap": True, "valign": "top"})

    # Set defaults first, then tune long text columns for each sheet
    for sheet_name, sheet_df in [("I&W_No", final_iw_no), ("I&W_Yes", final_iw_yes)]:
        worksheet = writer.sheets[sheet_name]
        worksheet.set_column(0, len(final_indicators.columns) - 1, 18)

        if "Explanation" in final_indicators.columns:
            _exp_idx = final_indicators.columns.get_loc("Explanation")
            worksheet.set_column(_exp_idx, _exp_idx, 100, wrap_fmt)

        if "Associated Groups" in final_indicators.columns:
            _ag_idx = final_indicators.columns.get_loc("Associated Groups")
            worksheet.set_column(_ag_idx, _ag_idx, 45, wrap_fmt)

output_path